# 02. Engine Evaluation Analysis

This notebook moves beyond game-level metadata and begins analyzing engine-evaluation patterns across Lichess games. The goal is to create careful evaluation-based features that help describe game volatility, turning points, upset patterns, and the relationship between rating advantage and in-game performance.

Unlike the previous SQL notebook, which focused on metadata such as game category, rating gaps, and termination type, this notebook uses move/evaluation-level columns to study how games develop internally.

## 1. Setup

This section defines the project paths and imports the libraries needed for evaluation-column inspection and feature engineering.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

CSV_PATH = RAW_DIR / "lichess_200k.csv"
PROCESSED_FILE = PROCESSED_DIR / "games_clean.csv"
EVAL_FEATURES_FILE = PROCESSED_DIR / "eval_features.csv"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

## 2. Load Clean Metadata and Raw Evaluation Columns

The clean metadata from Notebook 00 already contains the validated game-level fields and engineered features such as `winner`, `rating_favorite`, and rating-gap variables.

The raw CSV is still needed here because the engine-evaluation and clock columns were intentionally excluded from the metadata export. For this notebook, I load all available `Eval_ply_*` and `Clock_ply_*` columns while still avoiding unrelated raw columns.

Loading all available evaluation columns is important for final-state and termination analysis. A fixed early-ply window can be useful for early/mid-game volatility, but it should not be treated as the true final game state when many games continue beyond that range.

Evaluation and clock columns are loaded as strings first because some evaluation columns contain mixed formats: ordinary numeric evaluations such as `0.47` and forced-mate notation such as `#1` or `#-2`. Parsing is handled explicitly later.

In [2]:
# Load the cleaned game-level metadata from Notebook 00
metadata_df = pd.read_csv(PROCESSED_FILE)

# Read only the raw CSV header first so selected columns can be built safely
all_cols = pd.read_csv(CSV_PATH, nrows=0).columns.tolist()

# Select all available evaluation columns and keep them in ply order
eval_col_pairs = []

for col in all_cols:
    if col.startswith("Eval_ply_"):
        ply_number = int(col.split("_")[-1])
        eval_col_pairs.append((ply_number, col))

eval_col_pairs = sorted(eval_col_pairs)

eval_cols = []

for ply_number, col in eval_col_pairs:
    eval_cols.append(col)

# Select all available clock columns and keep them in ply order
clock_col_pairs = []

for col in all_cols:
    if col.startswith("Clock_ply_"):
        ply_number = int(col.split("_")[-1])
        clock_col_pairs.append((ply_number, col))

clock_col_pairs = sorted(clock_col_pairs)

clock_cols = []

for ply_number, col in clock_col_pairs:
    clock_cols.append(col)

raw_selected_cols = ["Index"] + eval_cols + clock_cols

dtype_overrides = {}

for col in eval_cols + clock_cols:
    dtype_overrides[col] = "string"

raw_eval_clock_df = pd.read_csv(
    CSV_PATH,
    usecols=raw_selected_cols,
    dtype=dtype_overrides,
    low_memory=False
)

# Match the raw game identifier to the cleaned metadata identifier
raw_eval_clock_df = raw_eval_clock_df.rename(columns={"Index": "game_id"})

metadata_df["game_id"] = metadata_df["game_id"].astype("int64")
raw_eval_clock_df["game_id"] = raw_eval_clock_df["game_id"].astype("int64")

# Combine clean metadata with selected raw evaluation/clock columns
# The cleaned metadata table is the base, so a left merge preserves all 200,000 games.
eval_df = metadata_df.merge(
    raw_eval_clock_df,
    on="game_id",
    how="left",
    validate="one_to_one"
)

eval_df.shape

(200000, 421)

## 3. Basic Structure Checks

Before cleaning the evaluation values, I check whether the merge preserved the expected row count and whether the loaded move-level columns were attached correctly.

In [3]:
# Basic row and column checks
print(f"Clean metadata shape: {metadata_df.shape}")
print(f"Raw eval/clock shape: {raw_eval_clock_df.shape}")
print(f"Merged eval_df shape: {eval_df.shape}")
print(f"Number of eval columns: {len(eval_cols)}")
print(f"Number of clock columns: {len(clock_cols)}")

Clean metadata shape: (200000, 21)
Raw eval/clock shape: (200000, 401)
Merged eval_df shape: (200000, 421)
Number of eval columns: 200
Number of clock columns: 200


In [4]:
# Inspect a small sample of evaluation and clock columns
eval_df[["game_id", "Category", "winner", "rating_favorite"] + eval_cols[:10]].head()

,game_id,Category,winner,rating_favorite,Eval_ply_1,Eval_ply_2,Eval_ply_3,Eval_ply_4,Eval_ply_5,Eval_ply_6,Eval_ply_7,Eval_ply_8,Eval_ply_9,Eval_ply_10
0,1,Blitz,Black,White favorite,0.12,0.37,0.23,0.15,0.21,0.46,0.36,0.38,0.48,0.36
1,2,Blitz,Black,Close rating,0.12,0.37,0.23,0.59,0.16,0.64,0.47,0.8,0.94,0.95
2,3,Rapid,White,Black favorite,0.12,0.46,0.28,0.9,0.69,0.97,0.91,0.94,0.85,0.77
3,4,Bullet,Black,Black favorite,0.12,0.2,0.21,0.08,0.15,0.22,0.08,0.37,0.25,0.85
4,5,Blitz,White,Close rating,0.25,0.62,0.64,0.62,0.47,0.43,0.38,0.98,0.63,0.98


## 4. Inspect Raw Evaluation Formats

The evaluation columns contain two different kinds of values:

- ordinary numeric evaluations, such as `0.47` or `-1.25`
- forced-mate notation, such as `#1` or `#-2`

These cannot be treated as the same scale. Numeric evaluations can be used for volatility and swing calculations, while mate notation should be handled separately as a forced-mate indicator.

In [5]:
# Inspect sample values from early evaluation columns
for col in eval_cols[:12]:
    print(f"{col}")
    print(eval_df[col].dropna().head(15).tolist())

Eval_ply_1
['0.12', '0.12', '0.12', '0.12', '0.25', '0.12', '0.24', '0.12', '0.12', '0.12', '0.12', '0.25', '0.25', '0.12', '0.25']
Eval_ply_2
['0.37', '0.37', '0.46', '0.2', '0.62', '0.31', '0.33', '0.2', '0.37', '0.2', '0.37', '0.15', '0.15', '0.2', '0.19']
Eval_ply_3
['0.23', '0.23', '0.28', '0.21', '0.64', '0.3', '0.09', '0.21', '-0.11', '0.0', '0.23', '0.22', '0.0', '0.21', '0.01']
Eval_ply_4
['0.15', '0.59', '0.9', '0.08', '0.62', '0.29', '0.44', '0.26', '0.86', '0.0', '0.59', '0.47', '0.14', '0.26', '0.37']
Eval_ply_5
['0.21', '0.16', '0.69', '0.15', '0.47', '0.27', '0.1', '0.17', '0.0', '-0.12', '0.4', '0.16', '0.0', '-0.37', '0.32']
Eval_ply_6
['0.46', '0.64', '0.97', '0.22', '0.43', '0.43', '0.38', '0.09', '0.16', '0.08', '0.59', '0.18', '0.56', '-0.26', '0.7']
Eval_ply_7
['0.36', '0.47', '0.91', '0.08', '0.38', '0.08', '0.4', '0.08', '0.0', '-0.38', '0.37', '0.0', '0.06', '-0.3', '0.31']
Eval_ply_8
['0.38', '0.8', '0.94', '0.37', '0.98', '0.16', '0.24', '0.38', '0.59', '0.0'

In [6]:
# Identify non-numeric strings in evaluation columns
non_numeric_examples = {}

for col in eval_cols:
    converted = pd.to_numeric(eval_df[col], errors="coerce")
    bad_values = eval_df.loc[
        eval_df[col].notna() & converted.isna(),
        col
    ].astype(str).unique()
    
    if len(bad_values) > 0:
        non_numeric_examples[col] = bad_values[:10]

len(non_numeric_examples), list(non_numeric_examples.items())[:5]

(198,
 [('Eval_ply_3', array(['#-1', '#-2'], dtype=object)),
  ('Eval_ply_4', array(['#1', '#2', '#-1'], dtype=object)),
  ('Eval_ply_5', array(['#1', '#-1', '#-2'], dtype=object)),
  ('Eval_ply_6',
   array(['#6', '#1', '#3', '#5', '#2', '#-1', '#4'], dtype=object)),
  ('Eval_ply_7',
   array(['#-7', '#5', '#-4', '#2', '#-1', '#-2', '#-6', '#3', '#4', '#-8'],
         dtype=object))])

## 5. Quantify Forced-Mate Notation

Forced-mate notation is meaningful chess information, not ordinary missing data. Before converting evaluations to numeric values, I measure how often mate notation appears across the loaded evaluation columns.

In [7]:
# Count forced-mate notation across evaluation columns
mate_counts = {}

for col in eval_cols:
    mate_mask = eval_df[col].astype("string").str.match(r"^#-?\d+$", na=False)
    count = mate_mask.sum()
    
    if count > 0:
        mate_counts[col] = count

len(mate_counts), list(mate_counts.items())[:10]

(198,
 [('Eval_ply_3', np.int64(7)),
  ('Eval_ply_4', np.int64(17)),
  ('Eval_ply_5', np.int64(9)),
  ('Eval_ply_6', np.int64(79)),
  ('Eval_ply_7', np.int64(25)),
  ('Eval_ply_8', np.int64(76)),
  ('Eval_ply_9', np.int64(44)),
  ('Eval_ply_10', np.int64(134)),
  ('Eval_ply_11', np.int64(91)),
  ('Eval_ply_12', np.int64(236))])

In [8]:
# Overall share of forced-mate notation among non-missing evaluation entries
total_mate_values = sum(mate_counts.values())
total_non_missing_eval_values = eval_df[eval_cols].notna().sum().sum()

mate_share = total_mate_values / total_non_missing_eval_values * 100

total_mate_values, total_non_missing_eval_values, round(mate_share, 3)

(np.int64(1028820), np.int64(12558096), np.float64(8.192))

The loaded evaluation columns contain both ordinary numeric engine evaluations and forced-mate notation such as `#1` or `#-2`.

Because forced-mate notation is not on the same numeric scale as ordinary evaluation values, I do not convert it into arbitrary large numbers. Instead, I separate the evaluation data into numeric evaluation values for volatility calculations and mate-related indicators for forced-mate patterns.

## 6. Convert Ordinary Evaluation Values to Numeric

For numeric evaluation analysis, ordinary engine evaluations are converted to numeric values. Forced-mate notation becomes missing in this numeric matrix, but it is preserved separately through mate-indicator features in the next section.

In [9]:
# Convert ordinary numeric evaluations while excluding forced-mate notation from numeric calculations
eval_numeric = eval_df[eval_cols].apply(pd.to_numeric, errors="coerce")

eval_numeric.dtypes.value_counts()

Float64    200
Name: count, dtype: int64

In [10]:
# Missingness after numeric conversion includes both true missing values and excluded mate notation
numeric_missing_share = eval_numeric.isna().mean().mul(100).round(2)

pd.DataFrame({
    "eval_column": numeric_missing_share.index,
    "missing_share_percent": numeric_missing_share.values
}).head(20)

,eval_column,missing_share_percent
0,Eval_ply_1,0.00
1,Eval_ply_2,0.00
2,Eval_ply_3,0.00
3,Eval_ply_4,0.01
4,Eval_ply_5,0.01
5,Eval_ply_6,0.05
6,Eval_ply_7,0.05
7,Eval_ply_8,0.10
8,Eval_ply_9,0.11
9,Eval_ply_10,0.20


In [11]:
# Later plies are expected to have more missing values because many games end before reaching them
pd.DataFrame({
    "eval_column": numeric_missing_share.index,
    "missing_share_percent": numeric_missing_share.values
}).tail(20)

,eval_column,missing_share_percent
180,Eval_ply_181,99.90
181,Eval_ply_182,99.91
182,Eval_ply_183,99.91
183,Eval_ply_184,99.91
184,Eval_ply_185,99.92
185,Eval_ply_186,99.92
186,Eval_ply_187,99.93
187,Eval_ply_188,99.93
188,Eval_ply_189,99.93
189,Eval_ply_190,99.94


## 7. Create Mate-Related Features

Mate notation is preserved through separate features. These features identify whether a game reached a forced-mate evaluation in the loaded evaluation columns, when it first appeared, and which side the mate notation favored.

In this notation, positive values such as `#3` indicate mate for White, while negative values such as `#-3` indicate mate for Black.

In [12]:
# Build mate masks from the raw string evaluation columns
eval_raw_str = eval_df[eval_cols].astype("string")

mate_mask = pd.DataFrame(index=eval_df.index)
white_mate_mask = pd.DataFrame(index=eval_df.index)
black_mate_mask = pd.DataFrame(index=eval_df.index)

for col in eval_cols:
    mate_mask[col] = eval_raw_str[col].str.match(r"^#-?\d+$", na=False)
    white_mate_mask[col] = eval_raw_str[col].str.match(r"^#\d+$", na=False)
    black_mate_mask[col] = eval_raw_str[col].str.match(r"^#-\d+$", na=False)

# Convert masks to NumPy arrays for row-wise calculations
mate_array = mate_mask.to_numpy()
white_mate_array = white_mate_mask.to_numpy()
black_mate_array = black_mate_mask.to_numpy()

# Extract ply numbers from column names like Eval_ply_1, Eval_ply_2, ...
ply_numbers_list = []

for col in eval_cols:
    ply_number = int(col.split("_")[-1])
    ply_numbers_list.append(ply_number)

ply_numbers = np.array(ply_numbers_list)

# Check whether each game had any mate evaluation
had_mate_eval = mate_array.any(axis=1)

# Find the first ply where mate notation appeared
first_mate_position = mate_array.argmax(axis=1)
first_mate_ply = np.where(had_mate_eval, ply_numbers[first_mate_position], np.nan)

# Store mate-related features at the game level
mate_features = pd.DataFrame({
    "game_id": eval_df["game_id"],
    "had_mate_eval": had_mate_eval,
    "first_mate_ply": first_mate_ply,
    "had_white_mate_eval": white_mate_array.any(axis=1),
    "had_black_mate_eval": black_mate_array.any(axis=1)
})

mate_features.head()


,game_id,had_mate_eval,first_mate_ply,had_white_mate_eval,had_black_mate_eval
0,1,True,55.00,False,True
1,2,False,NaN,False,False
2,3,True,63.00,False,True
3,4,False,NaN,False,False
4,5,True,44.00,True,False


In [13]:
# Quick summary of mate-related features
mate_summary = pd.DataFrame({
    "metric": [
        "games_with_any_mate_eval",
        "games_with_white_mate_eval",
        "games_with_black_mate_eval"
    ],
    "count": [
        mate_features["had_mate_eval"].sum(),
        mate_features["had_white_mate_eval"].sum(),
        mate_features["had_black_mate_eval"].sum()
    ]
})

mate_summary["share_of_games_percent"] = (
    mate_summary["count"] / len(mate_features) * 100
).round(2)

mate_summary

,metric,count,share_of_games_percent
0,games_with_any_mate_eval,122127,61.06
1,games_with_white_mate_eval,67285,33.64
2,games_with_black_mate_eval,60928,30.46


## 8. Create Numeric Evaluation Features

This section creates game-level features from the numeric evaluation matrix. These features summarize how much the ordinary engine evaluation changed during the loaded evaluation columns.

The most important features are:

- final available numeric evaluation
- absolute final available numeric evaluation
- number of available numeric evaluation points
- maximum absolute evaluation swing between consecutive available plies
- evaluation volatility, measured as the average absolute change between consecutive available plies
- total evaluation range across the loaded evaluation columns

Forced-mate notation is excluded from these numeric calculations because mate values are not on the same scale as ordinary numeric evaluations. Mate patterns are handled separately through the mate-related features created earlier.

These features are not direct proof of blunders, but they can be used as cautious proxies for volatility and candidate turning points.

In [14]:
# Consecutive evaluation changes across loaded evaluation columns
# Positive/negative direction is not needed here, so I use absolute changes.
eval_changes = eval_numeric.diff(axis=1).abs()

# Basic numeric evaluation features
max_eval_swing = eval_changes.max(axis=1)
eval_volatility = eval_changes.mean(axis=1)
final_available_eval = eval_numeric.ffill(axis=1).iloc[:, -1]
abs_final_available_eval = final_available_eval.abs()

numeric_eval_count = eval_numeric.notna().sum(axis=1)
numeric_eval_change_count = eval_changes.notna().sum(axis=1)

# Candidate large swing threshold in pawns; this is a proxy, not a confirmed blunder label
LARGE_SWING_THRESHOLD = 2.0
large_eval_swing_count = (eval_changes >= LARGE_SWING_THRESHOLD).sum(axis=1)

eval_features = pd.DataFrame({
    "game_id": eval_df["game_id"],
    "available_eval_count": numeric_eval_count,
    "available_eval_change_count": numeric_eval_change_count,
    "final_available_eval": final_available_eval,
    "abs_final_available_eval": abs_final_available_eval,
    "max_abs_eval": eval_numeric.abs().max(axis=1),
    "eval_range": eval_numeric.max(axis=1) - eval_numeric.min(axis=1),
    "max_eval_swing": max_eval_swing,
    "eval_volatility": eval_volatility,
    "large_eval_swing_count": large_eval_swing_count
})

eval_features.head()

,game_id,available_eval_count,available_eval_change_count,final_available_eval,abs_final_available_eval,max_abs_eval,eval_range,max_eval_swing,eval_volatility,large_eval_swing_count
0,1,55,53,-7.39,7.39,60.79,62.92,57.74,3.03,10
1,2,71,70,8.17,8.17,8.55,8.43,4.26,0.51,5
2,3,62,61,-26.55,26.55,35.44,37.51,10.90,1.46,14
3,4,71,70,15.69,15.69,15.87,15.79,5.68,0.90,12
4,5,43,42,20.85,20.85,21.45,21.20,10.73,0.77,3


In [15]:
# Combine numeric eval features with mate features and key metadata columns
candidate_context_cols = [
    "game_id",
    "Category",
    "winner",
    "rating_favorite",
    "rating_diff_white_minus_black",
    "abs_rating_diff",
    "Termination",
    "Result",
    "Opening",
    "ECO"
]

# Keep only context columns that exist in the cleaned metadata
context_cols = []

for col in candidate_context_cols:
    if col in eval_df.columns:
        context_cols.append(col)

analysis_features = (
    eval_df[context_cols]
    .merge(eval_features, on="game_id", how="left", validate="one_to_one")
    .merge(mate_features, on="game_id", how="left", validate="one_to_one")
)

# Create a game-level favorite outcome label for later SQL analysis
analysis_features["favorite_outcome"] = pd.NA

favorite_win_mask = (
    (analysis_features["winner"] == "White") & (analysis_features["rating_favorite"] == "White favorite")
) | (
    (analysis_features["winner"] == "Black") & (analysis_features["rating_favorite"] == "Black favorite")
)

upset_mask = (
    (analysis_features["winner"] == "White") & (analysis_features["rating_favorite"] == "Black favorite")
) | (
    (analysis_features["winner"] == "Black") & (analysis_features["rating_favorite"] == "White favorite")
)

analysis_features.loc[favorite_win_mask, "favorite_outcome"] = "Favorite won"
analysis_features.loc[upset_mask, "favorite_outcome"] = "Upset"

# Create the same rating-gap buckets used in the SQL metadata notebook
analysis_features["rating_gap_bucket"] = pd.NA

analysis_features.loc[
    (analysis_features["abs_rating_diff"] > 50) & (analysis_features["abs_rating_diff"] < 100),
    "rating_gap_bucket"
] = "51-99"

analysis_features.loc[
    (analysis_features["abs_rating_diff"] >= 100) & (analysis_features["abs_rating_diff"] < 200),
    "rating_gap_bucket"
] = "100-199"

analysis_features.loc[
    (analysis_features["abs_rating_diff"] >= 200) & (analysis_features["abs_rating_diff"] < 400),
    "rating_gap_bucket"
] = "200-399"

analysis_features.loc[
    analysis_features["abs_rating_diff"] >= 400,
    "rating_gap_bucket"
] = "400+"

analysis_features.head()


,game_id,Category,winner,rating_favorite,rating_diff_white_minus_black,abs_rating_diff,Termination,Result,Opening,ECO,available_eval_count,available_eval_change_count,final_available_eval,abs_final_available_eval,max_abs_eval,eval_range,max_eval_swing,eval_volatility,large_eval_swing_count,had_mate_eval,first_mate_ply,had_white_mate_eval,had_black_mate_eval,favorite_outcome,rating_gap_bucket
0,1,Blitz,Black,White favorite,65,65,Normal,0-1,Caro-Kann Defense,B15,55,53,-7.39,7.39,60.79,62.92,57.74,3.03,10,True,55.00,False,True,Upset,51-99
1,2,Blitz,Black,Close rating,16,16,Normal,0-1,Italian Game,C50,71,70,8.17,8.17,8.55,8.43,4.26,0.51,5,False,NaN,False,False,<NA>,<NA>
2,3,Rapid,White,Black favorite,-108,108,Normal,1-0,Philidor Defense #2,C41,62,61,-26.55,26.55,35.44,37.51,10.90,1.46,14,True,63.00,False,True,Upset,100-199
3,4,Bullet,Black,Black favorite,-80,80,Normal,0-1,Modern Defense,B06,71,70,15.69,15.69,15.87,15.79,5.68,0.90,12,False,NaN,False,False,Favorite won,51-99
4,5,Blitz,White,Close rating,19,19,Normal,1-0,Sicilian Defense: Loewenthal Variation,B32,43,42,20.85,20.85,21.45,21.20,10.73,0.77,3,True,44.00,True,False,<NA>,<NA>


## 9. Initial Feature Sanity Checks

Before answering analytical questions, I check whether the engineered features have reasonable distributions and whether missing values are expected.

In [16]:
analysis_features[
    [
        "available_eval_count",
        "available_eval_change_count",
        "final_available_eval",
        "abs_final_available_eval",
        "max_abs_eval",
        "eval_range",
        "max_eval_swing",
        "eval_volatility",
        "large_eval_swing_count"
    ]
].describe().T


,count,mean,std,min,25%,50%,75%,max
available_eval_count,"200,000.00",57.65,22.34,0.00,42.00,54.00,69.00,200.00
available_eval_change_count,"200,000.00",56.15,22.18,0.00,41.00,53.00,68.00,199.00
final_available_eval,"199,999.00",0.38,24.56,-153.22,-9.88,0.00,10.71,153.22
abs_final_available_eval,"199,999.00",16.24,18.42,0.00,5.35,10.32,18.57,153.22
max_abs_eval,"199,999.00",24.44,23.83,0.24,8.71,14.93,27.70,153.22
eval_range,"199,999.00",27.39,25.19,0.06,10.83,17.91,31.26,306.19
max_eval_swing,"199,999.00",16.15,18.00,0.06,5.73,9.24,16.18,210.61
eval_volatility,"199,999.00",1.28,0.94,0.02,0.66,1.00,1.59,14.89
large_eval_swing_count,"200,000.00",7.61,5.61,0.00,3.00,6.00,11.00,90.00


In [17]:
# Check feature missingness
analysis_features.isna().mean().mul(100).round(2).sort_values(ascending=False).head(20)

favorite_outcome                55.92
rating_gap_bucket               54.72
first_mate_ply                  38.94
Category                         0.00
rating_diff_white_minus_black    0.00
abs_rating_diff                  0.00
winner                           0.00
rating_favorite                  0.00
game_id                          0.00
Opening                          0.00
Result                           0.00
Termination                      0.00
ECO                              0.00
abs_final_available_eval         0.00
available_eval_count             0.00
available_eval_change_count      0.00
final_available_eval             0.00
max_eval_swing                   0.00
eval_range                       0.00
max_abs_eval                     0.00
dtype: float64

## 10. Starter Analytical Questions

After the evaluation features are validated, the next sections can use them to answer the main Notebook 02 questions:

1. Do upset games involve larger evaluation swings than favorite wins?
2. Are faster game categories associated with more volatile evaluations?
3. Are time-forfeit games usually lost, winning, or roughly equal before the flag?
4. Does rating gap still matter after accounting for evaluation volatility?
5. Can large evaluation swings act as cautious proxies for blunders or turning points?

The goal is not to claim perfect chess causality. The goal is to use engine-evaluation features carefully and explain their limitations clearly.

In [18]:
# Save engineered features for later SQL/visualization notebooks if needed
analysis_features.to_csv(EVAL_FEATURES_FILE, index=False)

EVAL_FEATURES_FILE

WindowsPath('../data/processed/eval_features.csv')

## 11. Validate the Engineered Feature Table

Before using the engineered features for SQL analysis, I reload the exported file and run a few basic validation checks. This confirms that the feature table was saved correctly, preserves the expected game-level structure, and contains reasonable missing-value patterns before moving into analysis.


In [19]:
features_check = pd.read_csv(EVAL_FEATURES_FILE)

features_check.shape

(200000, 25)

In [20]:
features_check.head()

,game_id,Category,winner,rating_favorite,rating_diff_white_minus_black,abs_rating_diff,Termination,Result,Opening,ECO,available_eval_count,available_eval_change_count,final_available_eval,abs_final_available_eval,max_abs_eval,eval_range,max_eval_swing,eval_volatility,large_eval_swing_count,had_mate_eval,first_mate_ply,had_white_mate_eval,had_black_mate_eval,favorite_outcome,rating_gap_bucket
0,1,Blitz,Black,White favorite,65,65,Normal,0-1,Caro-Kann Defense,B15,55,53,-7.39,7.39,60.79,62.92,57.74,3.03,10,True,55.00,False,True,Upset,51-99
1,2,Blitz,Black,Close rating,16,16,Normal,0-1,Italian Game,C50,71,70,8.17,8.17,8.55,8.43,4.26,0.51,5,False,NaN,False,False,NaN,NaN
2,3,Rapid,White,Black favorite,-108,108,Normal,1-0,Philidor Defense #2,C41,62,61,-26.55,26.55,35.44,37.51,10.90,1.46,14,True,63.00,False,True,Upset,100-199
3,4,Bullet,Black,Black favorite,-80,80,Normal,0-1,Modern Defense,B06,71,70,15.69,15.69,15.87,15.79,5.68,0.91,12,False,NaN,False,False,Favorite won,51-99
4,5,Blitz,White,Close rating,19,19,Normal,1-0,Sicilian Defense: Loewenthal Variation,B32,43,42,20.85,20.85,21.45,21.20,10.73,0.77,3,True,44.00,True,False,NaN,NaN


In [21]:
features_check.isna().mean().mul(100).round(2).sort_values(ascending=False)

favorite_outcome                55.92
rating_gap_bucket               54.72
first_mate_ply                  38.94
Category                         0.00
rating_diff_white_minus_black    0.00
abs_rating_diff                  0.00
winner                           0.00
rating_favorite                  0.00
game_id                          0.00
Opening                          0.00
Result                           0.00
Termination                      0.00
ECO                              0.00
abs_final_available_eval         0.00
available_eval_count             0.00
available_eval_change_count      0.00
final_available_eval             0.00
max_eval_swing                   0.00
eval_range                       0.00
max_abs_eval                     0.00
eval_volatility                  0.00
had_mate_eval                    0.00
large_eval_swing_count           0.00
had_black_mate_eval              0.00
had_white_mate_eval              0.00
dtype: float64

In [22]:
features_check.describe()

,game_id,rating_diff_white_minus_black,abs_rating_diff,available_eval_count,available_eval_change_count,final_available_eval,abs_final_available_eval,max_abs_eval,eval_range,max_eval_swing,eval_volatility,large_eval_swing_count,first_mate_ply
count,"200,000.00","200,000.00","200,000.00","200,000.00","200,000.00","199,999.00","199,999.00","199,999.00","199,999.00","199,999.00","199,999.00","200,000.00","122,127.00"
mean,"100,000.50",-0.53,78.60,57.65,56.15,0.38,16.24,24.44,27.39,16.15,1.28,7.61,55.59
std,"57,735.17",129.91,103.44,22.34,22.18,24.56,18.42,23.83,25.19,18.00,0.94,5.61,21.16
min,1.00,"-1,454.00",0.00,0.00,0.00,-153.22,0.00,0.24,0.06,0.06,0.02,0.00,3.00
25%,"50,000.75",-44.00,17.00,42.00,41.00,-9.88,5.35,8.71,10.83,5.73,0.66,3.00,41.00
50%,"100,000.50",0.00,43.00,54.00,53.00,0.00,10.32,14.93,17.91,9.24,1.00,6.00,52.00
75%,"150,000.25",43.00,98.00,69.00,68.00,10.71,18.57,27.70,31.26,16.18,1.59,11.00,67.00
max,"200,000.00","1,406.00","1,454.00",200.00,199.00,153.22,153.22,153.22,306.19,210.61,14.89,90.00,198.00


The exported feature table preserves the expected game-level structure and contains the engineered evaluation and mate-related features needed for the next analysis steps. Missing values in numeric evaluation features are expected because not every game has a numeric evaluation at every ply, and forced-mate notation was intentionally separated from ordinary numeric evaluations.


## 12. Load Engineered Features into SQLite

After exporting the engineered evaluation features, I load the feature table into SQLite so the game-level features can be analyzed with SQL. This keeps the analysis style consistent with the previous notebook while allowing grouped comparisons across favorite outcomes, rating-gap buckets, game categories, and termination types.

In [23]:
import sqlite3

EVAL_DB_PATH = PROCESSED_DIR / "eval_features.db"

conn = sqlite3.connect(EVAL_DB_PATH)

analysis_features.to_sql(
    "eval_features",
    conn,
    if_exists="replace",
    index=False
)

def run_query(query):
    return pd.read_sql_query(query, conn)

In [24]:
# Confirm that the SQLite table preserves the expected game-level structure
run_query("""
SELECT
    COUNT(*) AS total_rows
FROM eval_features;
""")

,total_rows
0,200000


In [25]:
# Inspect available SQL columns before writing analysis queries
run_query("""
PRAGMA table_info(eval_features);
""")

,cid,name,type,notnull,dflt_value,pk
0,0,game_id,INTEGER,0,None,0
1,1,Category,TEXT,0,None,0
2,2,winner,TEXT,0,None,0
3,3,rating_favorite,TEXT,0,None,0
4,4,rating_diff_white_minus_black,INTEGER,0,None,0
5,5,abs_rating_diff,INTEGER,0,None,0
6,6,Termination,TEXT,0,None,0
7,7,Result,TEXT,0,None,0
8,8,Opening,TEXT,0,None,0
9,9,ECO,TEXT,0,None,0


## 13. Favorite Wins vs Upsets: Evaluation Volatility

Notebook 01 showed that rating favorites usually win, but upsets still occur in a large share of decisive games. This section checks whether those upsets are associated with more unstable engine-evaluation patterns.

To do this, I compare favorite wins and upsets using the engineered evaluation features. If upsets have larger evaluation swings or higher evaluation volatility, that would suggest that favorites often lose in games where the position changes sharply rather than in smooth, stable games.

This analysis focuses only on decisive games with a clear rating favorite.


In [26]:
run_query("""
-- Favorite Wins vs Upsets: Evaluation Volatility
SELECT
    ef.favorite_outcome,
    COUNT(*) AS total_games,
    ROUND(AVG(ef.max_eval_swing), 2) AS avg_max_eval_swing,
    ROUND(AVG(ef.eval_volatility), 2) AS avg_eval_volatility,
    ROUND(AVG(ef.abs_final_available_eval), 2) AS avg_abs_final_eval,
    ROUND(
        100.0 * SUM(CASE
            WHEN ef.had_mate_eval = 1 THEN 1
            ELSE 0
        END) / COUNT(*),
        2
    ) AS mate_eval_rate,
    ROUND(
        AVG(CASE
            WHEN ef.had_mate_eval = 1 THEN ef.first_mate_ply
        END),
        2
    ) AS avg_first_mate_ply
FROM eval_features AS ef
WHERE ef.favorite_outcome IN ('Favorite won', 'Upset')
GROUP BY ef.favorite_outcome
ORDER BY
    CASE ef.favorite_outcome
        WHEN 'Favorite won' THEN 1
        WHEN 'Upset' THEN 2
    END;
""")

,favorite_outcome,total_games,avg_max_eval_swing,avg_eval_volatility,avg_abs_final_eval,mate_eval_rate,avg_first_mate_ply
0,Favorite won,54005,16.22,1.28,16.33,61.10,55.57
1,Upset,34154,15.98,1.27,16.17,61.21,55.36


The overall comparison does not show a meaningful difference between favorite wins and upsets. Favorite wins and upsets have almost identical average maximum evaluation swing, average evaluation volatility, final evaluation magnitude, mate-evaluation rate, and average first mate ply.

This is not the result I expected, but it is still useful. At the aggregate level, upsets are not simply explained by games being more evaluation-volatile. This suggests that the broad favorite-win versus upset comparison is too general to reveal a strong evaluation-based difference.


## 14. Evaluation Volatility by Rating-Gap Bucket and Favorite Outcome

The previous section compares favorite wins and upsets overall. This section checks whether that comparison changes inside rating-gap buckets.

Notebook 01 showed that rating-gap size is the strongest metadata signal for favorite reliability. Because of that, this query compares favorite wins and upsets within each rating-gap bucket to see whether larger upsets involve different evaluation patterns than closer-rating games.


In [27]:
run_query("""
-- Evaluation Volatility by Rating-Gap Bucket and Favorite Outcome
SELECT
    ef.rating_gap_bucket,
    ef.favorite_outcome,
    COUNT(*) AS total_games,
    ROUND(AVG(ef.max_eval_swing), 2) AS avg_max_eval_swing,
    ROUND(AVG(ef.eval_volatility), 2) AS avg_eval_volatility,
    ROUND(AVG(ef.abs_final_available_eval), 2) AS avg_abs_final_eval,
    ROUND(
        100.0 * SUM(CASE
            WHEN ef.had_mate_eval = 1 THEN 1
            ELSE 0
        END) / COUNT(*),
        2
    ) AS mate_eval_rate,
    ROUND(
        AVG(CASE
            WHEN ef.had_mate_eval = 1 THEN ef.first_mate_ply
        END),
        2
    ) AS avg_first_mate_ply
FROM eval_features AS ef
WHERE ef.favorite_outcome IN ('Favorite won', 'Upset')
    AND ef.rating_gap_bucket IS NOT NULL
GROUP BY
    ef.rating_gap_bucket,
    ef.favorite_outcome
ORDER BY
    CASE ef.rating_gap_bucket
        WHEN '400+' THEN 4
        WHEN '200-399' THEN 3
        WHEN '100-199' THEN 2
        WHEN '51-99' THEN 1
    END DESC,
    CASE ef.favorite_outcome
        WHEN 'Favorite won' THEN 1
        WHEN 'Upset' THEN 2
    END;
""")

,rating_gap_bucket,favorite_outcome,total_games,avg_max_eval_swing,avg_eval_volatility,avg_abs_final_eval,mate_eval_rate,avg_first_mate_ply
0,400+,Favorite won,3313,16.37,1.30,17.22,62.18,55.80
1,400+,Upset,889,15.91,1.29,15.40,61.87,55.67
2,200-399,Favorite won,9782,16.41,1.29,16.29,60.94,55.50
3,200-399,Upset,4130,16.04,1.28,16.10,61.53,55.54
4,100-199,Favorite won,18291,16.25,1.28,16.23,60.97,55.46
5,100-199,Upset,11309,15.95,1.26,16.21,61.01,55.35
6,51-99,Favorite won,22619,16.08,1.27,16.31,61.11,55.66
7,51-99,Upset,17826,15.99,1.27,16.19,61.24,55.32


Breaking the comparison down by rating-gap bucket still does not reveal a strong volatility difference between favorite wins and upsets. Even in larger rating-gap buckets, where an upset might be expected to require a sharper turning point, the average maximum swings and volatility values remain very close.

This means rating gap remains important for explaining how often favorites win, but these broad evaluation-volatility features do not clearly explain why some favorites lose.


## 15. Evaluation Volatility by Game Category

The previous sections compare evaluation behavior by favorite outcome and rating-gap bucket. This section shifts the comparison to game category.

Since Bullet, Blitz, Rapid, and Classical games are played under different time conditions, they may show different levels of evaluation instability. This query compares the engineered evaluation features across game categories.


In [28]:
run_query("""
-- Evaluation Volatility by Game Category
SELECT
    ef.Category,
    COUNT(*) AS total_games,
    ROUND(AVG(ef.max_eval_swing), 2) AS avg_max_eval_swing,
    ROUND(AVG(ef.eval_volatility), 2) AS avg_eval_volatility,
    ROUND(AVG(ef.abs_final_available_eval), 2) AS avg_abs_final_eval,
    ROUND(
        100.0 * SUM(CASE
            WHEN ef.had_mate_eval = 1 THEN 1
            ELSE 0
        END) / COUNT(*),
        2
    ) AS mate_eval_rate,
    ROUND(
        AVG(CASE
            WHEN ef.had_mate_eval = 1 THEN ef.first_mate_ply
        END),
        2
    ) AS avg_first_mate_ply
FROM eval_features AS ef
WHERE ef.Category IS NOT NULL
GROUP BY ef.Category
ORDER BY
    CASE ef.Category
        WHEN 'Bullet' THEN 1
        WHEN 'Blitz' THEN 2
        WHEN 'Rapid' THEN 3
        WHEN 'Classical' THEN 4
    END;
""")

,Category,total_games,avg_max_eval_swing,avg_eval_volatility,avg_abs_final_eval,mate_eval_rate,avg_first_mate_ply
0,Bullet,43253,16.13,1.27,16.26,61.31,55.57
1,Blitz,86333,16.17,1.28,16.26,60.94,55.52
2,Rapid,45216,16.15,1.28,16.24,61.18,55.73
3,Classical,25198,16.09,1.28,16.16,60.87,55.56


The category-level comparison is also very flat. Bullet, Blitz, Rapid, and Classical games have almost identical average maximum evaluation swings, average evaluation volatility, final evaluation magnitude, mate-evaluation rates, and average first mate ply.

This is an important contrast with Notebook 01. Game category strongly affected how games ended, especially through time forfeits, but it did not meaningfully separate average engine-evaluation volatility in this feature table.


## 16. Extreme Evaluation Swing Patterns

The previous sections use average evaluation features. Averages can hide skewed behavior, so this section shifts to tail behavior.

I first inspect the distribution of `max_eval_swing`, then use the 95th percentile as a data-driven threshold for unusually large evaluation swings. This avoids choosing an arbitrary cutoff and focuses the analysis on extreme position changes.


In [29]:
analysis_features["max_eval_swing"].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
)

count   199,999.00
mean         16.15
std          18.00
min           0.06
50%           9.24
75%          16.18
90%          49.65
95%          57.86
99%          70.53
max         210.61
Name: max_eval_swing, dtype: Float64

The distribution of `max_eval_swing` is strongly right-skewed. The median game has a much smaller maximum swing than the upper percentiles, while the top tail contains very large evaluation swings.

Because averages can hide this kind of tail behavior, I use the 95th percentile as a data-driven threshold for identifying unusually large evaluation swings instead of choosing an arbitrary cutoff.


In [30]:
# Use the 95th percentile as a data-driven extreme-swing threshold
extreme_swing_threshold = analysis_features["max_eval_swing"].quantile(0.95)

extreme_swing_threshold

np.float64(57.86099999999977)

In [31]:
query = f"""
-- Extreme Evaluation Swing Rate by Category and Favorite Outcome
SELECT
    ef.Category,
    ef.favorite_outcome,
    COUNT(*) AS total_games,
    SUM(CASE
        WHEN ef.max_eval_swing >= {extreme_swing_threshold} THEN 1
        ELSE 0
    END) AS extreme_swing_games,
    ROUND(
        100.0 * SUM(CASE
            WHEN ef.max_eval_swing >= {extreme_swing_threshold} THEN 1
            ELSE 0
        END) / COUNT(*),
        2
    ) AS extreme_swing_rate,
    ROUND(AVG(ef.max_eval_swing), 2) AS avg_max_eval_swing,
    ROUND(AVG(ef.eval_volatility), 2) AS avg_eval_volatility
FROM eval_features AS ef
WHERE ef.Category IS NOT NULL
    AND ef.favorite_outcome IN ('Favorite won', 'Upset')
    AND ef.max_eval_swing IS NOT NULL
GROUP BY
    ef.Category,
    ef.favorite_outcome
ORDER BY
    CASE ef.Category
        WHEN 'Bullet' THEN 1
        WHEN 'Blitz' THEN 2
        WHEN 'Rapid' THEN 3
        WHEN 'Classical' THEN 4
    END,
    CASE ef.favorite_outcome
        WHEN 'Favorite won' THEN 1
        WHEN 'Upset' THEN 2
    END;
"""

run_query(query)

,Category,favorite_outcome,total_games,extreme_swing_games,extreme_swing_rate,avg_max_eval_swing,avg_eval_volatility
0,Bullet,Favorite won,13046,627,4.81,16.28,1.28
1,Bullet,Upset,7744,384,4.96,15.94,1.27
2,Blitz,Favorite won,21469,1087,5.06,16.20,1.28
3,Blitz,Upset,14409,694,4.82,16.05,1.26
4,Rapid,Favorite won,12297,665,5.41,16.27,1.29
5,Rapid,Upset,7580,369,4.87,15.92,1.27
6,Classical,Favorite won,7192,371,5.16,16.06,1.28
7,Classical,Upset,4421,212,4.80,15.95,1.27


Even when focusing on extreme evaluation swings, the result remains surprisingly even. The extreme-swing rate stays close to 5% across both game category and favorite outcome groups, which is expected from a global 95th-percentile threshold but still useful for comparison.

The key point is that no category or favorite-outcome group is clearly overrepresented in the extreme-swing tail. This weakens the idea that upsets or faster time controls are mainly driven by more frequent large numeric evaluation collapses.


## 17. Final Available Evaluation State by Termination Type

The earlier metadata analysis showed that termination type is an important game-level outcome, especially because time forfeits are common in faster categories.

Now that the feature table uses all available evaluation columns, `final_available_eval` is a stronger approximation of the last numeric engine evaluation available for each game. This section compares normal endings and time forfeits from the winner's perspective.

Positive values from the winner's perspective mean the winner was ahead according to the final available numeric evaluation. Negative values mean the winner was behind despite eventually winning.


In [32]:
run_query("""
-- Final Available Evaluation State by Termination Type
WITH final_eval_state AS (
    SELECT
        ef.game_id,
        ef.Termination,
        ef.Category,
        ef.winner,
        ef.final_available_eval,
        ef.abs_final_available_eval,
        CASE
            WHEN ef.winner = 'White' THEN ef.final_available_eval
            WHEN ef.winner = 'Black' THEN -ef.final_available_eval
        END AS final_eval_from_winner_perspective
    FROM eval_features AS ef
    WHERE ef.winner IN ('White', 'Black')
        AND ef.final_available_eval IS NOT NULL
        AND ef.Termination IN ('Normal', 'Time forfeit')
)

SELECT
    fes.Termination,
    COUNT(*) AS total_games,
    ROUND(AVG(fes.final_eval_from_winner_perspective), 2) AS avg_final_eval_from_winner_perspective,
    ROUND(AVG(fes.abs_final_available_eval), 2) AS avg_abs_final_eval,
    ROUND(
        100.0 * SUM(CASE
            WHEN fes.final_eval_from_winner_perspective > 0.50 THEN 1
            ELSE 0
        END) / COUNT(*),
        2
    ) AS winner_ahead_rate,
    ROUND(
        100.0 * SUM(CASE
            WHEN fes.final_eval_from_winner_perspective BETWEEN -0.50 AND 0.50 THEN 1
            ELSE 0
        END) / COUNT(*),
        2
    ) AS roughly_equal_rate,
    ROUND(
        100.0 * SUM(CASE
            WHEN fes.final_eval_from_winner_perspective < -0.50 THEN 1
            ELSE 0
        END) / COUNT(*),
        2
    ) AS winner_behind_rate
FROM final_eval_state AS fes
GROUP BY fes.Termination
ORDER BY
    CASE fes.Termination
        WHEN 'Normal' THEN 1
        WHEN 'Time forfeit' THEN 2
    END;
""")

,Termination,total_games,avg_final_eval_from_winner_perspective,avg_abs_final_eval,winner_ahead_rate,roughly_equal_rate,winner_behind_rate
0,Normal,146716,-0.02,16.23,46.43,6.75,46.82
1,Time forfeit,48121,0.14,16.29,46.71,6.67,46.62


This termination comparison appears almost perfectly balanced between normal endings and time forfeits. However, this result depends on interpreting the sign of `final_available_eval`, so it needs a sanity check before making any directional claims about whether winners were ahead or behind.


In [33]:
run_query("""
SELECT
    ef.winner,
    COUNT(*) AS total_games,
    ROUND(AVG(ef.final_available_eval), 2) AS avg_final_available_eval,
    ROUND(
        100.0 * SUM(CASE
            WHEN ef.final_available_eval > 0.50 THEN 1
            ELSE 0
        END) / COUNT(*),
        2
    ) AS positive_eval_rate,
    ROUND(
        100.0 * SUM(CASE
            WHEN ef.final_available_eval BETWEEN -0.50 AND 0.50 THEN 1
            ELSE 0
        END) / COUNT(*),
        2
    ) AS roughly_equal_rate,
    ROUND(
        100.0 * SUM(CASE
            WHEN ef.final_available_eval < -0.50 THEN 1
            ELSE 0
        END) / COUNT(*),
        2
    ) AS negative_eval_rate
FROM eval_features AS ef
WHERE ef.winner IN ('White', 'Black')
    AND ef.final_available_eval IS NOT NULL
GROUP BY ef.winner;
""")


,winner,total_games,avg_final_available_eval,positive_eval_rate,roughly_equal_rate,negative_eval_rate
0,Black,94498,0.35,48.13,6.78,45.09
1,White,100351,0.37,47.82,6.68,45.50


The sanity check shows that the sign of `final_available_eval` does not clearly align with the game winner. White wins and Black wins have very similar positive and negative evaluation rates.

Because of this, I avoid interpreting positive values as White advantage and negative values as Black advantage in the final analysis. The safer features in this notebook are sign-independent features such as maximum evaluation swing, evaluation volatility, absolute final evaluation magnitude, and mate-notation indicators.


## 18. Final Reflection and Limitations

This notebook was honestly frustrating, but it taught an important lesson. I engineered several evaluation-based features expecting them to reveal clearer differences between favorite wins, upsets, rating-gap buckets, game categories, or termination types. That did not really happen.

The broad evaluation-volatility features were surprisingly similar across the main groups. Favorite wins and upsets looked almost the same. Rating-gap buckets did not reveal a strong volatility difference. Game categories also showed very similar evaluation patterns, even though Notebook 01 showed that categories differ strongly in how games end.

The main limitation is that these engineered features summarize broad numeric evaluation behavior. They are useful for exploring the data, but they are not automatically strong explanatory features. A technically interesting feature can still fail to explain the outcome clearly.

Another important limitation is the raw evaluation sign. The sanity check showed that positive and negative final evaluation values do not clearly align with White wins and Black wins. Without clearer documentation of the dataset's evaluation convention, it would be unsafe to make directional claims such as “the winner was ahead” or “the loser was behind.” For that reason, this notebook avoids strong winner-perspective claims based on the sign of the evaluation.

The big lesson is that feature engineering is hypothesis testing, not decoration. The goal is not to force every engineered feature to produce an impressive result. The goal is to build features carefully, test whether they explain something, and be honest when they do not. In this case, the metadata-level findings from Notebook 01 remained stronger and easier to interpret than the broad engine-evaluation summaries created here.

For the next notebook, I will use these results carefully: the visualization stage should emphasize the strongest findings from the metadata analysis, while treating the evaluation-feature work as an honest exploratory extension with clear limitations.
